# Fraud Detection: EDA, Data Quality and Cleaning

Issue 5. We check the quality of the four tables and decide how to clean them. We follow the same steps as in our earlier EDA and data cleaning projects: duplicates, data types, odd values, impossible values and missing values.

**Important: we check train and test.** Cleaning rules that are *fixed* (for example "convert this column to a date") are applied to **both** tables in the same way, otherwise the model sees different data at prediction time. Anything the data *teaches* us (for example a fill value such as a mean) is learned on **train only** and then reused on test. And we never drop rows from the **test** set, because Zindi needs a prediction for every test client.

In [1]:
# imports and display settings, so wide tables show all columns
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
DATA_DIR = Path("data")
# load the four Zindi tables (clients and invoices, train and test)

client_train = pd.read_csv(DATA_DIR / "client_train.csv")
# low_memory=False reads each column in one go, so pandas does not guess mixed types
client_test = pd.read_csv(DATA_DIR / "client_test.csv")
invoice_train = pd.read_csv(DATA_DIR / "invoice_train.csv", low_memory=False)
invoice_test = pd.read_csv(DATA_DIR / "invoice_test.csv", low_memory=False)

# names of the four consumption columns, reused in several checks below
consumption_cols = [f"consommation_level_{i}" for i in range(1, 5)]

## 1. Duplicates

In [2]:
# count rows that are exact copies of another row, in train and test
print("duplicate client rows (train, test):", client_train.duplicated().sum(), client_test.duplicated().sum())
print("duplicate invoice rows (train, test):", invoice_train.duplicated().sum(), invoice_test.duplicated().sum())

duplicate client rows (train, test): 0 0


duplicate invoice rows (train, test): 11 8


There are no duplicate clients. There are only a handful of exact duplicate invoice rows (11 in train, 8 in test). This is tiny compared to millions of rows. We have **not dropped them yet**: the team wants to take a closer look first (see the decision log at the end).

## 2. Data types: dates

In [3]:
# try to convert each date column with its expected format
for name, series, fmt in [
    ("client_train.creation_date", client_train["creation_date"], "%d/%m/%Y"),
    ("client_test.creation_date", client_test["creation_date"], "%d/%m/%Y"),
    ("invoice_train.invoice_date", invoice_train["invoice_date"], "%Y-%m-%d"),
    ("invoice_test.invoice_date", invoice_test["invoice_date"], "%Y-%m-%d"),
]:
    # errors="coerce" turns dates that cannot be parsed into NaT, so we can count them
    dates = pd.to_datetime(series, format=fmt, errors="coerce")
    print(f"{name}: unparsable = {dates.isna().sum()}, from {dates.min().date()} to {dates.max().date()}")

client_train.creation_date: unparsable = 0, from 1977-02-05 to 2019-09-10
client_test.creation_date: unparsable = 0, from 1977-02-05 to 2019-08-26


invoice_train.invoice_date: unparsable = 0, from 1977-06-09 to 2019-12-07
invoice_test.invoice_date: unparsable = 0, from 1977-07-13 to 2019-12-07


All dates convert without errors, so this is a safe fixed rule for train and test. Note the two different formats: `dd/mm/yyyy` for the clients and `yyyy-mm-dd` for the invoices.

Numeric codes like `disrict`, `client_catg`, `region`, `tarif_type` and `counter_code` are **categories**, not quantities. Averaging a district number makes no sense, so we treat them as categories later.

## 3. Odd values that block a conversion

Sometimes a column *looks* numeric, but a few strange entries stop pandas from converting it. In our weather exercise this was a `$` in the precipitation column. Here it is `counter_statue`, which should be a status from 0 to 5. Let's try to convert it to numbers:

In [4]:
# test if counter_statue can be turned into whole numbers, and show the error if not
for name, df in [("train", invoice_train), ("test", invoice_test)]:
    print(name, "dtype:", df["counter_statue"].dtype)
    try:
        df["counter_statue"].astype(int)
        print("  conversion to int works")
    except ValueError as error:
        print("  conversion to int FAILS:", error)

train dtype: str
  conversion to int FAILS: invalid literal for int() with base 10: 'A'
test dtype: int64
  conversion to int works


The error names the first value that breaks the conversion (`'A'`). Let's see all values outside 0 to 5:

In [5]:
# the valid statuses 0 to 5 as text, so text and numeric columns can be compared the same way
valid_status = [str(i) for i in range(6)]
# keep only rows whose status is outside 0 to 5
odd_train = invoice_train.loc[~invoice_train["counter_statue"].astype(str).isin(valid_status)]
odd_test = invoice_test.loc[~invoice_test["counter_statue"].astype(str).isin(valid_status)]

# how many odd rows there are, and which odd values appear in train
print("odd rows in train:", len(odd_train), "| in test:", len(odd_test))
odd_train["counter_statue"].value_counts()

odd rows in train:

 47 | in test: 0


counter_statue
46        14
A         13
618       12
769        6
269375     1
420        1
Name: count, dtype: int64

Only 47 rows (from 6 clients) have odd values, and **only in train**. None of these 6 clients is fraudulent, so the values do not look like a fraud signal. That is why the column is text in train but numeric in test. Options to discuss: set these values to missing (NaN) and treat them like an "unknown" status, or drop those rows in train. The rule must be the same for both tables.

Do these odd rows belong to fraudulent clients?

In [6]:
# compare the fraud share of the clients with odd values to the fraud share of all clients
odd_clients = odd_train["client_id"].unique()
odd_target = client_train.loc[client_train["client_id"].isin(odd_clients), "target"]
print("clients with odd counter_statue:", len(odd_clients), "| fraud share:", round(odd_target.mean(), 3))
print("overall fraud share:", round(client_train["target"].mean(), 3))

clients with odd counter_statue: 6 | fraud share: 0.0
overall fraud share: 0.056


## 4. Impossible or suspicious values

"Impossible" means a value that cannot be right in real life, like a negative amount of electricity used, or a meter reading that goes down.

**Consumption must not be negative.** You cannot use less than zero kWh or m³. We check all four consumption columns.

In [7]:
# count negative values per consumption column (impossible in real life)
for name, df in [("train", invoice_train), ("test", invoice_test)]:
    negatives = (df[consumption_cols] < 0).sum()
    print(name, "negative values per consumption column:", negatives.to_dict())

train negative values per consumption column: {'consommation_level_1': 0, 'consommation_level_2': 0, 'consommation_level_3': 0, 'consommation_level_4': 0}
test negative values per consumption column: {'consommation_level_1': 0, 'consommation_level_2': 0, 'consommation_level_3': 0, 'consommation_level_4': 0}


There are no negative consumption values. Good.

**The meter index should only go up.** Each invoice has the meter reading at the previous reading (`old_index`) and at this reading (`new_index`). A meter is like the odometer of a car: it only counts upwards, so `new_index` should be larger than or equal to `old_index`. If `new_index` is *smaller*, something is off. Examples are a meter that was replaced or reset, a counter that rolled over after 99999, a typing error, or manipulation (which is exactly what we are looking for!). Let's look:

In [8]:
# count rows where the meter reading goes down or stays the same
for name, df in [("train", invoice_train), ("test", invoice_test)]:
    backward = df["new_index"] < df["old_index"]
    print(f"{name}: new_index < old_index in {backward.sum()} rows ({backward.mean():.3%}), equal in {(df['new_index'] == df['old_index']).sum()} rows")

# show a few backward rows to see if they look like rollovers or small errors
backward_train = invoice_train[invoice_train["new_index"] < invoice_train["old_index"]]
backward_train[["client_id", "old_index", "new_index", "consommation_level_1", "counter_type"]].sample(8, random_state=42)

train: new_index < old_index in 2264 rows (0.051%), equal in 472683 rows
test: new_index < old_index in 1054 rows (0.054%), equal in 203986 rows


,client_id,old_index,new_index,consommation_level_1,counter_type
3437931,train_Client_7182,9907,214,307,ELEC
2482797,train_Client_45761,98599,1537,800,ELEC
703216,train_Client_118946,99276,1218,1942,ELEC
859762,train_Client_123305,96520,2225,400,ELEC
120674,train_Client_103261,99635,755,800,ELEC
3018255,train_Client_60349,97491,10582,1200,ELEC
1670013,train_Client_23494,97688,65,200,ELEC
3240648,train_Client_6643,9827,742,800,ELEC


Some rows look like a **rollover**: the old index is close to the maximum (for example 99,260) and the new one starts again near zero (778), which is normal for a meter display with five digits. Others only go back by a tiny amount (19,772 to 19,770), which looks like an error or a correction. Such rows are not safe to delete, because they might be fraud signals. We keep them and later create a feature such as "number of backward invoices per client".

**Months between readings.** `months_number` should be a small number such as 1, 2, 4, 8 or 12:

In [9]:
# range of months_number and how many rows have very long or zero gaps
for name, df in [("train", invoice_train), ("test", invoice_test)]:
    print(name, "months_number: min", df["months_number"].min(), "max", df["months_number"].max(),
          "| rows above 24:", (df["months_number"] > 24).sum(), "| above 120:", (df["months_number"] > 120).sum(), "| rows equal 0:", (df["months_number"] == 0).sum())

# the most common gaps between readings
invoice_train["months_number"].value_counts().head(10)

train months_number: min 0 max 636624 | rows above 24: 3528 | above 120: 1411 | rows equal 0: 2
test months_number: min 0 max 990688 | rows above 24: 1809 | above 120: 887 | rows equal 0: 9


months_number
4     3680478
8      278772
2      270909
1      113266
12      54328
6       34678
16      11550
3        7286
10       6993
5        3945
Name: count, dtype: int64

Most invoices are 1 to 16 months, but there is a long tail. Values between 17 and 120 months are mostly even numbers (18, 20, 24, 28, 36, 40, ...), which look like real long gaps between meter readings. Above 120 months (10 years) the values become impossible: 1,411 train rows and 887 test rows, up to hundreds of thousands of months. **Team decision: we keep all `months_number` values** (no limit), see the decision log at the end.

**Other suspicious values:**

In [10]:
# the most common counter_coefficient values and how often it is 0
print("counter_coefficient value counts (train):")
print(invoice_train["counter_coefficient"].value_counts().head(8).to_dict())
print("rows with coefficient 0:", (invoice_train["counter_coefficient"] == 0).sum(), "(train),", (invoice_test["counter_coefficient"] == 0).sum(), "(test)")

# share of invoices with no consumption at all, and the largest level 1 value
for name, df in [("train", invoice_train), ("test", invoice_test)]:
    all_zero = (df[consumption_cols].sum(axis=1) == 0).mean()
    print(f"{name}: invoices with zero consumption in all four levels: {all_zero:.1%}")
    print(f"{name}: biggest consumption_level_1:", df["consommation_level_1"].max())

counter_coefficient value counts (train):
{1: 4475102, 2: 886, 3: 321, 40: 197, 30: 137, 0: 46, 6: 30, 4: 12}
rows with coefficient 0: 46 (train), 30 (test)


train: invoices with zero consumption in all four levels: 10.4%
train: biggest consumption_level_1: 999910


test: invoices with zero consumption in all four levels: 10.4%
test: biggest consumption_level_1: 110858


- A `counter_coefficient` of 0 is odd (it should multiply the consumption), but it is rare.
- About one in ten invoices has zero consumption on all levels. This can be real (empty house) or a fraud signal, so we **keep** these rows.
- The largest consumption values (close to a million) are extreme outliers. We look at them in the visual EDA before deciding.
- `reading_remarque` has more values in train than in test, so we should check this before using it as a feature.

In [11]:
# compare the reading_remarque codes in train and test, a feature must have the same codes in both
print("reading_remarque train:", sorted(invoice_train["reading_remarque"].unique()))
print("reading_remarque test: ", sorted(invoice_test["reading_remarque"].unique()))
invoice_train["reading_remarque"].value_counts()

reading_remarque train: [np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(203), np.int64(207), np.int64(413)]
reading_remarque test:  [np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


reading_remarque
6      2230939
9      1416992
8       828123
7          661
203         15
413         12
207          6
5            1
Name: count, dtype: int64

## 5. Missing values

In [12]:
# total number of NaN values per table
for name, df in [("client_train", client_train), ("client_test", client_test), ("invoice_train", invoice_train), ("invoice_test", invoice_test)]:
    print(name, "missing values:", int(df.isna().sum().sum()))

client_train missing values: 0
client_test missing values: 0


invoice_train missing values: 0
invoice_test missing values: 0


No table has real missing values (NaN). But missing values can be hidden as placeholders such as 0 or -1, so the "zero consumption" and "coefficient 0" findings above matter. The cleaning rules above (odd `counter_statue`, extreme `months_number`) will create missing values on purpose.

## Summary: cleaning decisions to agree on

| Finding | Train | Test | Proposed rule (to discuss) |
| --- | --- | --- | --- |
| Duplicate invoice rows | 11 | 8 | Open: take a closer look first |
| Dates are text | yes | yes | Convert both tables the same way |
| `counter_statue` odd values (`A`, `46`, ...) | 47 rows | none | Set to NaN in both, then treat as "unknown" |
| Negative consumption | none | none | Nothing to do |
| `new_index` < `old_index` | 2,264 rows | 1,054 rows | Keep, feature "backward steps per client" in feature engineering |
| Impossible `months_number` (above 120) | 1,411 rows | 887 rows | Keep all values, decide in feature engineering |
| `counter_coefficient` = 0 | 46 rows | 30 rows | Keep, check later |
| `reading_remarque` rare codes only in train | 8 values | 4 values | Keep 6 to 9, group the rare codes into 0 ("other") |
| Zero consumption invoices | ~10% | ~10% | Keep, possible fraud signal |

Rule of thumb: fixed rules apply to train **and** test; things learned from data are fitted on train only; test rows are never dropped.

## 6. Apply the cleaning

The rules live in [cleaning.py](cleaning.py), so everybody in the team can use the exact same code. Fixed rules are applied to **train and test** in exactly the same way. No rows are dropped.

| Rule | Train | Test |
| --- | --- | --- |
| Column `disrict` renamed to `district` (typo in the Zindi file) | yes | yes |
| Dates converted to dates | yes | yes |
| `counter_statue` outside 0 to 5 set to NaN | yes | yes |
| `months_number` kept as it is (no limit) | yes | yes |
| `reading_remarque` codes 5, 203, 207, 413 grouped into 0 ("other") | yes | yes (no effect) |
| Duplicate invoice rows dropped | no (still open) | no |
| Negative index steps, zero consumption, coefficient 0 | kept | kept |

In [13]:
# apply the shared cleaning rules from cleaning.py to train and test in the same way
from cleaning import clean_clients, clean_invoices

client_train_clean = clean_clients(client_train)
client_test_clean = clean_clients(client_test)
invoice_train_clean = clean_invoices(invoice_train)
invoice_test_clean = clean_invoices(invoice_test)

### Check the result

In [14]:
# check that cleaning dropped no rows
print("invoice rows train:", len(invoice_train), "->", len(invoice_train_clean))
print("invoice rows test: ", len(invoice_test), "->", len(invoice_test_clean))
print("clients train:", len(client_train_clean), "| clients test:", len(client_test_clean))
# check that every client still has at least one invoice
print("clients still having invoices (train, test):",
      client_train_clean["client_id"].isin(invoice_train_clean["client_id"]).all(),
      client_test_clean["client_id"].isin(invoice_test_clean["client_id"]).all())

invoice rows train: 4476749 -> 4476749
invoice rows test:  1939730 -> 1939730
clients train: 135493 | clients test: 58069
clients still having invoices (train, test): True True


In [15]:
# check data types, NaN counts and remaining values after cleaning, for train and test
for name, df in [("train", invoice_train_clean), ("test", invoice_test_clean)]:
    print(f"--- {name}")
    print("dtypes:", df[["invoice_date", "counter_statue", "months_number"]].dtypes.astype(str).to_dict())
    print("missing counter_statue:", df["counter_statue"].isna().sum(), "| missing months_number:", df["months_number"].isna().sum())
    print("months_number max:", df["months_number"].max(), "| counter_statue values:", sorted(df["counter_statue"].dropna().unique().astype(int)))
    print("reading_remarque values:", sorted(df["reading_remarque"].unique()))

--- train
dtypes: {'invoice_date': 'datetime64[us]', 'counter_statue': 'float64', 'months_number': 'int64'}
missing counter_statue: 47 | missing months_number: 0
months_number max: 636624 | counter_statue values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
reading_remarque values: [np.int64(0), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
--- test
dtypes: {'invoice_date': 'datetime64[us]', 'counter_statue': 'float64', 'months_number': 'int64'}
missing counter_statue: 0 | missing months_number: 0
months_number max: 990688 | counter_statue values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
reading_remarque values: [np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


Both tables now have the same data types. No rows were removed, all clients still have invoices, and `counter_statue` contains only valid values or NaN (which models like LightGBM and XGBoost can handle). The impossible `months_number` values are kept on purpose.

### Save the cleaned data

The cleaned tables are saved as CSV files in `data/clean/` (ignored by git, like all data), so teammates can load them directly.

Still to do in the next steps, because they are feature engineering and not cleaning: aggregate invoices per client, create features such as the number of backward index steps, and handle outliers after we look at the distributions.

## Team decisions and open questions

The team discussed the points below on 2026-10-20. All decisions are also written down in [protocol.md](protocol.md).

| Topic | Decision | Status |
| --- | --- | --- |
| Odd `counter_statue` values | Keep the rows and set the values to NaN | Decided |
| `months_number` | Keep all values, no maximum. Decide in feature engineering | Decided |
| Duplicate invoices | Take a closer look again | Open |
| Meter index going backward | Keep. Look closer in feature engineering. Hypothesis: the number of backward steps per client indicates fraud | Decided |
| Outliers | Keep all rows, no capping (recommendation below) | Decided |
| `reading_remarque` | Keep 6 to 9, group rare codes into 0 ("other") | To confirm with the team |
| Presentation | Say that we kept the outliers on purpose, because they are what we are looking for | Noted |

### Odd `counter_statue` values (`A`, `46`, `618`, ...)

47 invoice rows in train (from 6 clients) have a value outside the valid range 0 to 5. In test there are none. We **keep the rows and set the value to NaN**: it is only 47 of 4.5 million rows, all other information in the invoice is kept, train and test follow the same rule, and LightGBM and XGBoost handle NaN.

### `months_number`

The column holds the number of months between two meter readings. 99.8% of the invoices cover 1 to 16 months. Values from 17 to 120 months are mostly even numbers (18, 20, 24, 28, 36, ...) and look like real long gaps. Above 120 months the values are impossible (1,411 train rows and 887 test rows, up to hundreds of thousands of months).

**Decision: we keep all values and set no maximum.** We do not want to lose information that may help us find the most fraudulent clients. In feature engineering we decide how to use the column. For reference: clients with a value above 24 are fraudulent in 4.8% of the cases, compared to 5.6% overall, so a link is not visible yet.

### Duplicate invoices

11 exact duplicate rows in train (10 clients) and 8 in test (3 clients). **Open:** the team looks at them again before deciding. Until then nothing is dropped. Our earlier thought was to drop them in both tables, because a double-entered invoice is an error and inflates features such as the invoice count per client.

### Meter index going backward

2,264 train rows and 1,054 test rows have `new_index` smaller than `old_index`. Some are normal rollovers after 99,999, others only step back by a few units. **We keep them.** In feature engineering we look at them closer and add this to our hypotheses: the **number of backward steps per client** may be an indicator for fraud.

### `reading_remarque`: what do the numbers mean?

Zindi describes the column as "notes taken by the STEG agent during a visit", for example a bad score if the counter shows something wrong. Neither the data page nor the forum explains the numeric codes, so we treat the column as a **category**, not as a quantity.

Fraud rate per client who has at least one invoice with this remark (train only; overall rate 5.6%):

| `reading_remarque` | Clients | Fraudulent clients | Fraud rate |
| --- | --- | --- | --- |
| 6 | 119,781 | 7,151 | 6.0% |
| 8 | 95,591 | 6,927 | 7.2% |
| 9 | 116,596 | 7,426 | 6.4% |
| 7 | 478 | 1 | 0.2% |
| 5 | 1 | 0 | 0% |
| 203 | 2 | 0 | 0% |
| 207 | 1 | 0 | 0% |
| 413 | 1 | 0 | 0% |

Codes 6, 8 and 9 cover almost all clients and are close to the overall rate. Code 7 is rare but stands out (1 of 478 clients is fraudulent) and also appears in test. Codes 5, 203, 207 and 413 cover only 5 clients and never appear in test. **Current choice in [cleaning.py](cleaning.py):** keep 6 to 9, group the rare codes into `0` = "other" (no effect on test).

### Outliers

A few values are far away from the others. `consommation_level_1` has a median of 274, 99% of the values are below 2,192, but the maximum is 999,910 (only two rows are above 100,000). The options are to **keep** the values (tree models only look at thresholds), to **cap** them at a limit such as the 99.9th percentile (computed on train only), or to **transform** them with a logarithm (`log1p`).

Extreme values may be exactly what we are looking for: clients with at least one `consommation_level_1` above 10,000 (541 clients in train) are fraudulent in 11.8% of the cases, compared to 5.6% overall.

**Decision: we follow the recommendation.**
- Keep all rows and do not cap for now. Tree models handle outliers well, and capping could hide the fraud signal.
- When we aggregate per client, also use the **median** and the **max**, not only the mean.
- For the logistic regression baseline, use a **log transform** of the consumption features.
- Look at the distributions first. Only if a handful of values are obvious typing errors (for example 999,910), we cap at a train-only percentile.

**For the presentation:** we kept the outliers on purpose, because extreme values are what we are looking for.

In [16]:
# create data/clean/ if it does not exist yet
clean_dir = DATA_DIR / "clean"
clean_dir.mkdir(exist_ok=True)

# save the cleaned tables without the pandas index column
client_train_clean.to_csv(clean_dir / "client_train_clean.csv", index=False)
client_test_clean.to_csv(clean_dir / "client_test_clean.csv", index=False)
invoice_train_clean.to_csv(clean_dir / "invoice_train_clean.csv", index=False)
invoice_test_clean.to_csv(clean_dir / "invoice_test_clean.csv", index=False)

# list the saved files and the client columns as a quick check
print(sorted(path.name for path in clean_dir.iterdir()))
print(client_train_clean.columns.tolist())

['client_test_clean.csv', 'client_train_clean.csv', 'invoice_test_clean.csv', 'invoice_train_clean.csv']
['district', 'client_id', 'client_catg', 'region', 'creation_date', 'target']
